# ***Aoun (عون) - GP2***

# IMPORTS

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings

In [3]:
from collections import Counter
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, recall_score, precision_score, f1_score
)
from xgboost import XGBClassifier

In [4]:
warnings.filterwarnings("ignore")


# PHASE 1 — EDA

In [5]:
from google.colab import drive
drive.mount('/content/drive')

ModuleNotFoundError: No module named 'google'

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/DataSets/cancer_patient_data.csv')

In [ ]:
df['Genetic Risk'] = pd.to_numeric(df['Genetic Risk'], errors='coerce')
df['Obesity']      = pd.to_numeric(df['Obesity'], errors='coerce')

In [ ]:
TARGET = "Level"

In [ ]:
print(f"Shape: {df.shape}")

Shape: (743728, 26)


In [ ]:
# Missing values
print("\nMissing values:")
print(df.isnull().sum())


Missing values:
index                           0
Patient Id                      0
Age                         22402
Gender                      37348
Air Pollution                   0
Alcohol use                     0
Dust Allergy                    0
OccuPational Hazards            0
Genetic Risk                    1
chronic Lung Disease            0
Balanced Diet                   0
Obesity                         1
Smoking                     59409
Passive Smoker                  0
Chest Pain                  14887
Coughing of Blood               0
Fatigue                         0
Weight Loss                     0
Shortness of Breath             0
Wheezing                        0
Swallowing Difficulty           0
Clubbing of Finger Nails        0
Frequent Cold                   0
Dry Cough                       0
Snoring                         0
Level                           0
dtype: int64


In [ ]:
# Target distribution
print("\nTarget distribution:")
print(df[TARGET].value_counts())


Target distribution:
Level
High      268737
Medium    247191
Low       227800
Name: count, dtype: int64


In [ ]:
df[TARGET].value_counts().plot(kind="bar")
plt.title("Risk distribution")
plt.savefig("01_class_distribution.png")
plt.close()

In [ ]:
# Drop unwanted columns
drop_cols = [TARGET]
for col in ["index", "Patient Id"]:
    if col in df.columns:
        drop_cols.append(col)

In [ ]:
# Correlation (Used a sample of 10k for speed on large data)
numeric_df = df.select_dtypes(include=[np.number])
corr = numeric_df.sample(n=min(10000, len(df))).corr()
sns.heatmap(corr)
plt.savefig("02_correlation_heatmap.png")
plt.close()

In [ ]:
# Encode target
le = LabelEncoder()
df["Level_encoded"] = le.fit_transform(df[TARGET])

In [ ]:
print("EDA done")

EDA done


# PHASE 2 — ML

In [ ]:
# Clip Age Outliers
if 'Age' in df.columns:
    df['Age'] = df['Age'].clip(1, 100)

In [ ]:
TARGET_ENC = "Level_encoded"
X = df.drop(columns=drop_cols + [TARGET_ENC]).select_dtypes(include=[np.number])
y = df[TARGET_ENC]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
# Handling Missing Values
# We use 'median' because it's more robust to outliers

print("Imputing missing values...")
imputer = SimpleImputer(strategy='median')
X_train = imputer.fit_transform(X_train)
X_test = imputer.transform(X_test)

Imputing missing values...


In [ ]:
# Scaling

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

In [ ]:
# Models
models = {
    "RF": RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1),
    "XGB": XGBClassifier(eval_metric="mlogloss", n_jobs=-1),
    "LR": LogisticRegression(max_iter=1000)
}

In [ ]:
results = {}

In [ ]:
print("\n" + "="*50)
print("MODEL RESULTS")
print("="*50)

for name, model in models.items():
    print(f"\nTraining {name}...")

    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    acc  = accuracy_score(y_test, pred)
    rec  = recall_score(y_test, pred, average="macro")
    prec = precision_score(y_test, pred, average="macro")
    f1   = f1_score(y_test, pred, average="macro")

    results[name] = {
        "model": model,
        "accuracy": acc,
        "recall": rec,
        "precision": prec,
        "f1": f1
    }

    print(f"Accuracy : {acc:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"F1 Score : {f1:.4f}")

    print("\nClassification Report:")
    print(classification_report(y_test, pred))


MODEL RESULTS

Training RF...
Accuracy : 0.9201
Recall   : 0.9195
Precision: 0.9201
F1 Score : 0.9198

Classification Report:
              precision    recall  f1-score   support

           0       0.92      0.93      0.92     53748
           1       0.92      0.91      0.91     45560
           2       0.92      0.92      0.92     49438

    accuracy                           0.92    148746
   macro avg       0.92      0.92      0.92    148746
weighted avg       0.92      0.92      0.92    148746


Training XGB...
Accuracy : 0.9201
Recall   : 0.9195
Precision: 0.9201
F1 Score : 0.9198

Classification Report:
              precision    recall  f1-score   support

           0       0.92      0.93      0.92     53748
           1       0.92      0.91      0.91     45560
           2       0.92      0.92      0.92     49438

    accuracy                           0.92    148746
   macro avg       0.92      0.92      0.92    148746
weighted avg       0.92      0.92      0.92    148746

In [ ]:
print("\n" + "="*50)
print("FINAL COMPARISON")
print("="*50)

for name, r in results.items():
    print(f"{name}: Recall={r['recall']:.4f}, Accuracy={r['accuracy']:.4f}, F1={r['f1']:.4f}")


FINAL COMPARISON
RF: Recall=0.9195, Accuracy=0.9201, F1=0.9198
XGB: Recall=0.9195, Accuracy=0.9201, F1=0.9198
LR: Recall=0.9031, Accuracy=0.9040, F1=0.9031


In [ ]:
# Best model (by recall)
best_name = max(results, key=lambda x: results[x]["recall"])
best_model = results[best_name]["model"]
print(f"\n Best model: {best_name}")


 Best model: RF


In [ ]:
# Confusion matrix for the best model
cm = confusion_matrix(y_test, best_model.predict(X_test))
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title(f"Confusion Matrix: {best_name}")
plt.savefig("04_confusion_matrix.png")
plt.close()

# Saving everything for Phase 3 (Deployment)
joblib.dump(best_model, "aoun_model.pkl")
joblib.dump(scaler, "aoun_scaler.pkl")
joblib.dump(le, "aoun_label_encoder.pkl")
joblib.dump(imputer, "aoun_imputer.pkl")

['aoun_imputer.pkl']

In [ ]:
# Save feature names
pd.Series(list(X.columns)).to_csv("aoun_features.csv", index=False)

In [ ]:
# Save results to CSV
results_df = pd.DataFrame(results).T.drop(columns=["model"])
results_df.to_csv("model_results.csv")